In [2]:
#Working but division is broken

# import os, sys
# import random
# from datetime import datetime, timedelta

# # 1. ENVIRONMENT SETUP
# os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
# os.environ['PYSPARK_PYTHON'] = sys.executable

# from pyspark.sql import SparkSession
# import pyspark.sql.functions as F  # ONLY USE THIS
# from pyspark.sql.types import *

# spark = SparkSession.builder \
#     .appName('Synthetic') \
#     .master('local[*]') \
#     .getOrCreate()

# # 2. NEIGHBOURHOOD MAPPING
# neighbourhood_pairs = [
#     ("Junction-Wallace Emerson (171)","Dovercourt-Wallace Emerson-Junction (93)"),
#     ("Bay-Cloverhill (169)","Bay Street Corridor (76)"),
#     ("Little Portugal (84)","Little Portugal (84)"),
#     ("Junction Area (90)","Junction Area (90)"),
#     ("West Queen West (162)","Niagara (82)"),
#     ("St Lawrence-East Bayfront-The Islands (166)","Waterfront Communities-The Island (77)"),
#     ("Downtown Yonge East (168)","Church-Yonge Corridor (75)"),
#     ("Regent Park (72)","Regent Park (72)"),
#     ("Moss Park (73)","Moss Park (73)"),
#     ("South Riverdale (70)","South Riverdale (70)"),
#     ("Oakwood Village (107)","Oakwood Village (107)"),
#     ("Palmerston-Little Italy (80)","Palmerston-Little Italy (80)"),
#     ("Harbourfront-CityPlace (165)","Waterfront Communities-The Island (77)"),
#     ("Kensington-Chinatown (78)","Kensington-Chinatown (78)")
# ]

# # 3. LOAD REFERENCE DATA
# ref_df = spark.read.csv(
#     "/home/vboxuser/Big Data Systems Design/Final Project/Bike/Bikes_Cleaned.csv",
#     header=True,
#     inferSchema=True
# )

# def get_values(col_name):
#     return [row[0] for row in ref_df.select(col_name).distinct().collect() if row[0] is not None]

# categories = {k: get_values(k) for k in [
#     "PRIMARY_OFFENCE", "LOCATION_TYPE", "PREMISES_TYPE", 
#     "BIKE_MAKE", "BIKE_MODEL", "BIKE_TYPE", 
#     "BIKE_SPEED", "BIKE_COLOUR", "BIKE_COST"
# ]}

# # 4. DEFINE EXPLICIT SCHEMA
# schema = StructType([
#     StructField("NEIGHBOURHOOD_158", StringType(), True),
#     StructField("NEIGHBOURHOOD_140", StringType(), True),
#     StructField("HOOD_158", StringType(), True),
#     StructField("HOOD_140", StringType(), True),
#     StructField("PRIMARY_OFFENCE", StringType(), True),
#     StructField("LOCATION_TYPE", StringType(), True),
#     StructField("PREMISES_TYPE", StringType(), True),
#     StructField("BIKE_MAKE", StringType(), True),
#     StructField("BIKE_MODEL", StringType(), True),
#     StructField("BIKE_TYPE", StringType(), True),
#     StructField("BIKE_SPEED", StringType(), True),
#     StructField("BIKE_COLOUR", StringType(), True),
#     StructField("BIKE_COST", DoubleType(), True),
#     StructField("STATUS", StringType(), True),
#     StructField("OCC_YEAR", IntegerType(), True),
#     StructField("OCC_MONTH", IntegerType(), True),
#     StructField("OCC_DAY", IntegerType(), True),
#     StructField("OCC_DOW", StringType(), True),
#     StructField("OCC_DOY", IntegerType(), True),
#     StructField("OCC_HOUR", IntegerType(), True),
#     StructField("REPORT_YEAR", IntegerType(), True),
#     StructField("REPORT_MONTH", IntegerType(), True),
#     StructField("REPORT_DAY", IntegerType(), True),
#     StructField("REPORT_DOW", StringType(), True),
#     StructField("REPORT_DOY", IntegerType(), True),
#     StructField("REPORT_HOUR", IntegerType(), True),
#     StructField("LAT_WGS84", DoubleType(), True),
#     StructField("LONG_WGS84", DoubleType(), True),
#     StructField("x", DoubleType(), True),
#     StructField("y", DoubleType(), True),
#     StructField("DIVISION", IntegerType(), True),
#     StructField("OCCURENCE_PERIOD", StringType(), True)
# ])

# # 5. GENERATE DATA
# def generate_data(n=25000):
#     rows = []
#     now = datetime.now()

#     for _ in range(n):
#         hood158, hood140 = random.choice(neighbourhood_pairs)
#         occ_time = now.replace(hour=random.randint(0, 23), minute=random.randint(0, 59), second=random.randint(0, 59))
#         report_time = occ_time + timedelta(hours=random.randint(1, 72))
        
#         occ_hour = occ_time.hour
#         if 6 <= occ_hour <= 12: period = "Morning"
#         elif 12 < occ_hour <= 17: period = "Afternoon"
#         elif 17 < occ_hour <= 20: period = "Evening"
#         else: period = "Night"

#         # Now Python's round() will work because we removed 'from pyspark.sql.functions import *'
#         row = (
#             hood158, 
#             hood140,
#             hood158.split("(")[-1].replace(")", ""),
#             hood140.split("(")[-1].replace(")", ""),
#             random.choice(categories["PRIMARY_OFFENCE"]),
#             random.choice(categories["LOCATION_TYPE"]),
#             random.choice(categories["PREMISES_TYPE"]),
#             random.choice(categories["BIKE_MAKE"]),
#             random.choice(categories["BIKE_MODEL"]),
#             random.choice(categories["BIKE_TYPE"]),
#             str(random.choice(categories["BIKE_SPEED"])), 
#             random.choice(categories["BIKE_COLOUR"]),
#             float(random.choice(categories["BIKE_COST"])), 
#             "STOLEN",
#             occ_time.year, occ_time.month, occ_time.day, occ_time.strftime("%A"),
#             occ_time.timetuple().tm_yday, occ_hour,
#             report_time.year, report_time.month, report_time.day, report_time.strftime("%A"),
#             report_time.timetuple().tm_yday, report_time.hour,
#             round(random.uniform(43.60, 43.75), 6),
#             round(random.uniform(-79.55, -79.30), 6),
#             round(random.uniform(-8850000, -8750000), 2),
#             round(random.uniform(5400000, 5450000), 2),
#             random.randint(11, 55),
#             period
#         )
#         rows.append(row)

#     return spark.createDataFrame(rows, schema=schema)

# # 6. EXECUTE AND SAVE
# synthetic_df = generate_data(25000)
# synthetic_df.write.mode("overwrite").csv("synthetic_stream_data", header=True)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/24 19:53:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/24 19:53:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/24 19:53:33 WARN TaskSetManager: Stage 29 contains a task of very large size (1047 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

In [3]:
spark.stop()

In [5]:
# With divisions fixed

import os, sys
import random
from datetime import datetime, timedelta

# 1. ENVIRONMENT SETUP
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PYSPARK_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *

# Initialize Spark
spark = SparkSession.builder \
    .appName('TorontoBikeSyntheticData') \
    .master('local[*]') \
    .getOrCreate()

# 2. CONFIGURATION & MAPPING
neighbourhood_pairs = [
    ("Junction-Wallace Emerson (171)","Dovercourt-Wallace Emerson-Junction (93)"),
    ("Bay-Cloverhill (169)","Bay Street Corridor (76)"),
    ("Little Portugal (84)","Little Portugal (84)"),
    ("Junction Area (90)","Junction Area (90)"),
    ("West Queen West (162)","Niagara (82)"),
    ("St Lawrence-East Bayfront-The Islands (166)","Waterfront Communities-The Island (77)"),
    ("Downtown Yonge East (168)","Church-Yonge Corridor (75)"),
    ("Regent Park (72)","Regent Park (72)"),
    ("Moss Park (73)","Moss Park (73)"),
    ("South Riverdale (70)","South Riverdale (70)"),
    ("Oakwood Village (107)","Oakwood Village (107)"),
    ("Palmerston-Little Italy (80)","Palmerston-Little Italy (80)"),
    ("Harbourfront-CityPlace (165)","Waterfront Communities-The Island (77)"),
    ("Kensington-Chinatown (78)","Kensington-Chinatown (78)")
]

# Specific Toronto Divisions as requested
valid_divisions = ["D11", "D12", "D13", "D14", "D51", "D52", "D53", "D54", "D55"]

# 3. LOAD REFERENCE DATA
ref_path = "/home/vboxuser/Big Data Systems Design/Final Project/Bike/Bikes_Cleaned.csv"
ref_df = spark.read.csv(ref_path, header=True, inferSchema=True)

def get_values(col_name):
    # Extract actual values from the Row objects
    return [row[0] for row in ref_df.select(col_name).distinct().collect() if row[0] is not None]

categories = {k: get_values(k) for k in [
    "PRIMARY_OFFENCE", "LOCATION_TYPE", "PREMISES_TYPE", 
    "BIKE_MAKE", "BIKE_MODEL", "BIKE_TYPE", 
    "BIKE_SPEED", "BIKE_COLOUR", "BIKE_COST"
]}

# 4. DEFINE EXPLICIT SCHEMA
# Note: DIVISION is now StringType to support "D11", etc.
schema = StructType([
    StructField("NEIGHBOURHOOD_158", StringType(), True),
    StructField("NEIGHBOURHOOD_140", StringType(), True),
    StructField("HOOD_158", StringType(), True),
    StructField("HOOD_140", StringType(), True),
    StructField("PRIMARY_OFFENCE", StringType(), True),
    StructField("LOCATION_TYPE", StringType(), True),
    StructField("PREMISES_TYPE", StringType(), True),
    StructField("BIKE_MAKE", StringType(), True),
    StructField("BIKE_MODEL", StringType(), True),
    StructField("BIKE_TYPE", StringType(), True),
    StructField("BIKE_SPEED", StringType(), True),
    StructField("BIKE_COLOUR", StringType(), True),
    StructField("BIKE_COST", DoubleType(), True),
    StructField("STATUS", StringType(), True),
    StructField("OCC_YEAR", IntegerType(), True),
    StructField("OCC_MONTH", IntegerType(), True),
    StructField("OCC_DAY", IntegerType(), True),
    StructField("OCC_DOW", StringType(), True),
    StructField("OCC_DOY", IntegerType(), True),
    StructField("OCC_HOUR", IntegerType(), True),
    StructField("REPORT_YEAR", IntegerType(), True),
    StructField("REPORT_MONTH", IntegerType(), True),
    StructField("REPORT_DAY", IntegerType(), True),
    StructField("REPORT_DOW", StringType(), True),
    StructField("REPORT_DOY", IntegerType(), True),
    StructField("REPORT_HOUR", IntegerType(), True),
    StructField("LAT_WGS84", DoubleType(), True),
    StructField("LONG_WGS84", DoubleType(), True),
    StructField("x", DoubleType(), True),
    StructField("y", DoubleType(), True),
    StructField("DIVISION", StringType(), True),
    StructField("OCCURENCE_PERIOD", StringType(), True)
])

# 5. GENERATE DATA
def generate_data(n=25000):
    rows = []
    now = datetime.now()

    for _ in range(n):
        # Pick valid neighbourhood combo
        hood158, hood140 = random.choice(neighbourhood_pairs)
        
        # Timing logic
        occ_time = now.replace(hour=random.randint(0, 23), minute=random.randint(0, 59), second=random.randint(0, 59))
        report_time = occ_time + timedelta(hours=random.randint(1, 72))
        
        occ_hour = occ_time.hour
        if 6 <= occ_hour <= 12: period = "Morning"
        elif 12 < occ_hour <= 17: period = "Afternoon"
        elif 17 < occ_hour <= 20: period = "Evening"
        else: period = "Night"

        # Build Tuple (Matches Schema Order)
        row = (
            hood158, 
            hood140,
            hood158.split("(")[-1].replace(")", ""),
            hood140.split("(")[-1].replace(")", ""),
            random.choice(categories["PRIMARY_OFFENCE"]),
            random.choice(categories["LOCATION_TYPE"]),
            random.choice(categories["PREMISES_TYPE"]),
            random.choice(categories["BIKE_MAKE"]),
            random.choice(categories["BIKE_MODEL"]),
            random.choice(categories["BIKE_TYPE"]),
            str(random.choice(categories["BIKE_SPEED"])), 
            random.choice(categories["BIKE_COLOUR"]),
            float(random.choice(categories["BIKE_COST"])), 
            "STOLEN",
            occ_time.year, occ_time.month, occ_time.day, occ_time.strftime("%A"),
            occ_time.timetuple().tm_yday, occ_hour,
            report_time.year, report_time.month, report_time.day, report_time.strftime("%A"),
            report_time.timetuple().tm_yday, report_time.hour,
            round(random.uniform(43.60, 43.75), 6),
            round(random.uniform(-79.55, -79.30), 6),
            round(random.uniform(-8850000, -8750000), 2),
            round(random.uniform(5400000, 5450000), 2),
            random.choice(valid_divisions), # Fixed: returns D11, D12, etc.
            period
        )
        rows.append(row)

    return spark.createDataFrame(rows, schema=schema)

# 6. EXECUTE AND SAVE
synthetic_df = generate_data(25000)

# Writing to folder (Overwrite mode)
# repartition(1) will force it into a single CSV file inside the folder
synthetic_df.repartition(1).write.mode("overwrite").csv("synthetic_stream_data", header=True)

print("Data generation complete. Check folder: synthetic_stream_data")


26/03/24 20:07:00 WARN TaskSetManager: Stage 29 contains a task of very large size (1052 KiB). The maximum recommended task size is 1000 KiB.
[Stage 29:===========================================>              (3 + 1) / 4]

Data generation complete. Check folder: synthetic_stream_data
